# Pipeline 2: Resident Reintegration Readiness

## 1. Problem Framing

**Business Question:** Which residents are ready for reintegration — and what factors most predict (and explain) successful outcomes?

**Who cares:** Social workers and program directors. From the case: *"The founders worry about girls falling through the cracks... they need to know which girls are progressing and which are struggling, which interventions are actually working, and when a resident might be ready for reintegration or at risk of regression."* This pipeline directly supports the core operational mission.

**Dual Approach — Predictive AND Explanatory:**

We build two distinct models (Ch. 1 textbook distinction):

1. **Explanatory (OLS Logistic via statsmodels):** *What factors most contribute to successful reintegration?* Coefficients, odds ratios, and p-values are the primary output. We want to understand the data-generating process so social workers can intervene more effectively.
2. **Predictive (Gradient Boosting):** *Which current residents are most likely to successfully reintegrate?* Out-of-sample AUC-ROC is the primary concern. Used to generate a readiness score shown on resident profiles.

**Target Variable:** `reintegration_success` — 1 if `reintegration_status == 'Completed'`, 0 otherwise (In Progress or not started). Only residents with a recorded reintegration_status are included.

**Important caveat on dataset size:** With ~60 residents, statistical power is very limited. All findings are directional — treat as hypotheses to validate as data grows.

**Success Metrics:**
- Explanatory: McFadden's R², odds ratios, p-values, VIF (multicollinearity check)
- Predictive: AUC-ROC (cross-validated), confusion matrix

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
sys.path.insert(0, '..')

from pyLibrary import (
    univariate, unistats, bivariate, correlation_heatmap,
    missing_data_diagnostics, missing_data_clean, basic_wrangling,
    transform_skew, cap_outliers_iqr,
    build_preprocessor, make_pipeline_for_model, split_data,
    eval_classification, plot_roc_curve, plot_confusion_matrix,
    cross_validate_model, plot_learning_curve, plot_validation_curve,
    tune_grid, select_features_rfe,
    permutation_importance_report, feature_importance_plot,
    plot_logit_coefficients, ols_summary,
    compute_vif, remove_high_vif
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR     = Path('../data/lighthouse_csv_v7')
RANDOM_STATE = 42

In [ ]:
# ── Load all related tables ───────────────────────────────────────────────────
residents   = pd.read_csv(DATA_DIR / 'residents.csv',
                          parse_dates=['date_of_admission','date_enrolled','date_closed'])
education   = pd.read_csv(DATA_DIR / 'education_records.csv',   parse_dates=['record_date'])
health      = pd.read_csv(DATA_DIR / 'health_wellbeing_records.csv', parse_dates=['record_date'])
recordings  = pd.read_csv(DATA_DIR / 'process_recordings.csv',  parse_dates=['session_date'])
incidents   = pd.read_csv(DATA_DIR / 'incident_reports.csv',    parse_dates=['incident_date'])
plans       = pd.read_csv(DATA_DIR / 'intervention_plans.csv',  parse_dates=['target_date'])
visitations = pd.read_csv(DATA_DIR / 'home_visitations.csv',    parse_dates=['visit_date'])

for name, tbl in [('residents', residents), ('education', education), ('health', health),
                   ('recordings', recordings), ('incidents', incidents),
                   ('plans', plans), ('visitations', visitations)]:
    print(f'{name}: {tbl.shape}')

In [ ]:
# ── Missing data diagnostics for residents table (Ch. 7) ─────────────────────
missing_data_diagnostics(residents, verbose=True)

In [ ]:
# ── Feature engineering: aggregate per-resident across 6 tables ───────────────

# Education
edu_agg = education.groupby('resident_id').agg(
    avg_edu_progress   = ('progress_percent',  'mean'),
    avg_attendance     = ('attendance_rate',    'mean'),
    edu_months         = ('record_date',        'count'),
    edu_completed_pct  = ('completion_status',  lambda x: (x == 'Completed').mean())
).reset_index()

# Health
health_agg = health.groupby('resident_id').agg(
    avg_health         = ('general_health_score', 'mean'),
    avg_nutrition      = ('nutrition_score',       'mean'),
    avg_sleep          = ('sleep_quality_score',   'mean'),
    avg_energy         = ('energy_level_score',    'mean'),
    health_months      = ('record_date',           'count')
).reset_index()

# Counseling sessions
rec_agg = recordings.groupby('resident_id').agg(
    session_count      = ('recording_id',          'count'),
    avg_duration_min   = ('session_duration_minutes', 'mean'),
    progress_ratio     = ('progress_noted',
                          lambda x: x.astype(str).str.lower().eq('true').mean()),
    concern_ratio      = ('concerns_flagged',
                          lambda x: x.astype(str).str.lower().eq('true').mean())
).reset_index()

# Incidents
inc_agg = incidents.groupby('resident_id').agg(
    incident_count     = ('incident_id',  'count'),
    high_sev_incidents = ('severity',     lambda x: (x == 'High').sum())
).reset_index()

# Intervention plans
plan_agg = plans.groupby('resident_id').agg(
    plan_count         = ('plan_id',   'count'),
    plan_completion_rate = ('status',  lambda x: (x == 'Completed').mean())
).reset_index()

# Visitations
vis_agg = visitations.groupby('resident_id').agg(
    visit_count        = ('visitation_id',     'count'),
    safety_concern_rate = ('safety_concerns_noted',
                           lambda x: x.astype(str).str.lower().eq('true').mean()),
    favorable_rate     = ('visit_outcome',     lambda x: (x == 'Favorable').mean())
).reset_index()

print('All aggregations complete.')

In [ ]:
# ── Join all features onto residents (reproducible pipeline, Ch. 7) ───────────
df = residents.copy()
for agg in [edu_agg, health_agg, rec_agg, inc_agg, plan_agg, vis_agg]:
    df = df.merge(agg, on='resident_id', how='left')

# Risk level: ordinal encode
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
df['initial_risk_enc'] = df['initial_risk_level'].map(risk_map)
df['current_risk_enc'] = df['current_risk_level'].map(risk_map)
df['risk_improvement'] = df['initial_risk_enc'] - df['current_risk_enc']  # positive = improved

# Length of stay in months
df['los_months'] = (
    df['length_of_stay'].str.extract(r'(\d+)\s*Years').astype(float) * 12 +
    df['length_of_stay'].str.extract(r'(\d+)\s*months').fillna(0).astype(float)
)

# Number of adverse sub-categories
sub_cat_cols = [c for c in df.columns if c.startswith('sub_cat_')]
df['num_adverse_cats'] = df[sub_cat_cols].apply(
    lambda row: sum(str(v).lower() == 'true' for v in row), axis=1
)

# Target
df['reintegration_success'] = (df['reintegration_status'] == 'Completed').astype(int)

# Fill NaN aggregates with 0 (no records in a table)
fill_cols = ['avg_edu_progress','avg_attendance','edu_months','edu_completed_pct',
             'avg_health','avg_nutrition','avg_sleep','avg_energy','health_months',
             'session_count','avg_duration_min','progress_ratio','concern_ratio',
             'incident_count','high_sev_incidents',
             'plan_count','plan_completion_rate',
             'visit_count','safety_concern_rate','favorable_rate']
for col in fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

model_df = df[df['reintegration_status'].notna()].copy()
print(f'Modeling dataset: {len(model_df)} residents | {model_df["reintegration_success"].sum()} completed')

In [ ]:
# ── Univariate stats (Ch. 6) ──────────────────────────────────────────────────
FEATURES = [
    'initial_risk_enc', 'risk_improvement', 'los_months', 'num_adverse_cats',
    'session_count', 'progress_ratio', 'concern_ratio',
    'incident_count', 'high_sev_incidents',
    'plan_completion_rate', 'plan_count',
    'visit_count', 'favorable_rate', 'safety_concern_rate',
    'avg_health', 'avg_edu_progress', 'avg_attendance',
    'reintegration_success'
]
feat_df = model_df[FEATURES].copy()

print('=== Univariate Statistics ===')
unistats(feat_df)

In [ ]:
# ── Bivariate analysis: every feature vs. target (Ch. 8) ─────────────────────
corr_results = bivariate(feat_df, target='reintegration_success')
print('\nPearson correlations with reintegration success:')
print(corr_results)

In [ ]:
# ── Correlation heatmap (Ch. 8) ───────────────────────────────────────────────
corr_matrix = correlation_heatmap(feat_df)

In [ ]:
# ── Skew transformation (Ch. 7) ───────────────────────────────────────────────
skew_targets = ['los_months', 'session_count', 'incident_count',
                'visit_count', 'plan_count']
feat_df_t = transform_skew(feat_df, features=skew_targets, suffix='_t')
for col in skew_targets:
    feat_df[col] = feat_df_t[col + '_t']

feat_df = cap_outliers_iqr(feat_df, cols=skew_targets)
print('Transformations applied.')

## 3. Modeling & Feature Selection

In [ ]:
# ── VIF check for multicollinearity (required for explanatory model, Ch. 10) ──
num_feats = [c for c in FEATURES if c != 'reintegration_success']
X_num_only = feat_df[num_feats].copy()
for col in X_num_only.columns:
    X_num_only[col] = X_num_only[col].fillna(X_num_only[col].median())

print('=== VIF before removal ===')
vif_before = compute_vif(X_num_only)
print(vif_before.to_string(index=False))

# Iteratively remove highest VIF features until all < 10
X_low_vif = remove_high_vif(X_num_only, threshold=10.0)
print(f'\nFeatures remaining after VIF removal: {list(X_low_vif.columns)}')

In [ ]:
# ── Explanatory model: OLS Logistic via ols_summary (statsmodels) (Ch. 9-11) ──
# We use the VIF-clean feature set for valid causal inference
exp_df = X_low_vif.copy()
exp_df['reintegration_success'] = feat_df['reintegration_success'].values

# ols_summary auto one-hot-encodes categoricals and fits OLS
# For logistic, we use statsmodels Logit directly below
X_exp = sm.add_constant(X_low_vif.astype(float))
logit_res = sm.Logit(feat_df['reintegration_success'].values, X_exp).fit(disp=False)
print(logit_res.summary())

In [ ]:
# ── Odds ratios from explanatory logit ───────────────────────────────────────
odds_df = pd.DataFrame({
    'feature':    ['const'] + list(X_low_vif.columns),
    'coef':       logit_res.params,
    'odds_ratio': np.exp(logit_res.params),
    'p_value':    logit_res.pvalues
}).sort_values('p_value')

print(f"McFadden's R²: {1 - logit_res.llf / logit_res.llnull:.3f}")
print('\nOdds ratios (features with p < 0.20):')
print(odds_df[odds_df['feature'] != 'const'].to_string(index=False))

In [ ]:
# ── Visualize odds ratios ─────────────────────────────────────────────────────
sig = odds_df[(odds_df['feature'] != 'const') & (odds_df['p_value'] < 0.25)]\
      .sort_values('odds_ratio')

if len(sig) > 0:
    colors = ['steelblue' if v > 1 else 'salmon' for v in sig['odds_ratio']]
    plt.figure(figsize=(9, max(4, len(sig)*0.45 + 1)))
    plt.barh(sig['feature'], sig['odds_ratio'], color=colors, alpha=0.8)
    plt.axvline(1, color='black', linestyle='--', linewidth=1)
    plt.xlabel('Odds Ratio (>1 = more likely to succeed)')
    plt.title('Odds Ratios — Reintegration Success (Explanatory Logit, p<0.25)')
    plt.tight_layout()
    plt.show()
else:
    print('No features passed the p<0.25 threshold — dataset too small for stable estimates.')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# ── Predictive model: train/test split (Ch. 11, 15) ──────────────────────────
X_train, X_test, y_train, y_test = split_data(
    feat_df, target='reintegration_success',
    test_size=0.2, random_state=RANDOM_STATE, stratify=True
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Success rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

In [ ]:
# ── Build leakage-free pipelines (Ch. 11) ────────────────────────────────────
lr_pipe = make_pipeline_for_model(X_train, LogisticRegression(max_iter=1000, C=0.1, random_state=RANDOM_STATE))
gb_pipe = make_pipeline_for_model(X_train, GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=RANDOM_STATE))

print('=== 5-Fold CV — AUC-ROC ===')
for name, pipe in [('Logistic Regression (explanatory)', lr_pipe),
                   ('Gradient Boosting (predictive)',     gb_pipe)]:
    print(f'\n{name}:')
    cross_validate_model(pipe, X_train, y_train, cv=5, scoring='roc_auc', stratified=True)

In [ ]:
# ── Learning curve (Ch. 15) ───────────────────────────────────────────────────
plot_learning_curve(gb_pipe, X_train, y_train, cv=5, scoring='roc_auc')

In [ ]:
# ── Hyperparameter tuning (Ch. 15) ───────────────────────────────────────────
param_grid = {
    'model__n_estimators':  [100, 200],
    'model__max_depth':     [2, 3, 4],
    'model__learning_rate': [0.05, 0.1]
}
best_gb, gs = tune_grid(gb_pipe, param_grid, X_train, y_train,
                         cv=5, scoring='roc_auc')

In [ ]:
# ── RFECV feature selection on preprocessed array (Ch. 16) ───────────────────
preprocessor_fit, num_cols, cat_cols = build_preprocessor(X_train)
X_train_prep = preprocessor_fit.fit_transform(X_train)
X_test_prep  = preprocessor_fit.transform(X_test)

feature_names_out = (
    num_cols +
    list(preprocessor_fit.named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(cat_cols))
    if cat_cols else num_cols
)

_, selected_features = select_features_rfe(
    X_train_prep, y_train, feature_names_out,
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    cv=5, scoring='roc_auc'
)

In [ ]:
# ── MDI feature importance (Ch. 14, 16) ──────────────────────────────────────
best_gb.fit(X_train, y_train)
feature_importance_plot(
    best_gb.named_steps['model'],
    feature_names_out,
    top_n=15,
    title='MDI Feature Importance — Reintegration Readiness'
)

In [ ]:
# ── Permutation importance on TEST data (Ch. 16) ──────────────────────────────
pfi = permutation_importance_report(
    best_gb, X_test_prep, y_test, feature_names_out,
    n_repeats=10, scoring='roc_auc', top_n=15
)
print('\nTop 10 by permutation importance (unbiased, test-set):')
print(pfi.head(10).to_string(index=False))

In [ ]:
# ── Logistic regression coefficient plot (Ch. 13) ────────────────────────────
lr_pipe.fit(X_train, y_train)
plot_logit_coefficients(
    lr_pipe, top_n=20,
    title='Logistic Regression Coefficients — Reintegration Success'
)

## 4. Evaluation & Interpretation

In [ ]:
# ── Final evaluation on hold-out test set (Ch. 15) ────────────────────────────
print('=== Gradient Boosting (Predictive) ===')
results_gb = eval_classification(
    'Gradient Boosting (Tuned)', best_gb,
    X_train, y_train, X_test, y_test
)

print('\n=== Logistic Regression (Explanatory) ===')
results_lr = eval_classification(
    'Logistic Regression', lr_pipe,
    X_train, y_train, X_test, y_test, fit=False
)

In [ ]:
plot_roc_curve(best_gb, X_test, y_test,
               title='ROC Curve — Reintegration Readiness (Gradient Boosting)')

In [ ]:
plot_confusion_matrix(best_gb, X_test, y_test,
                      labels=[0, 1],
                      title='Confusion Matrix — Reintegration Readiness')

In [ ]:
# ── Readiness scores for all active residents ─────────────────────────────────
best_gb.fit(feat_df.drop(columns=['reintegration_success']),
            feat_df['reintegration_success'])

model_df = model_df.copy()
score_X  = feat_df.drop(columns=['reintegration_success'])
model_df['readiness_score'] = best_gb.predict_proba(score_X)[:, 1]
model_df['readiness_tier']  = pd.cut(
    model_df['readiness_score'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Not Ready', 'Approaching', 'Ready']
)

print('Readiness distribution:')
print(model_df['readiness_tier'].value_counts())

active = model_df[model_df['case_status'] == 'Active'].sort_values(
    'readiness_score', ascending=False
)[['resident_id','case_category','current_risk_level','los_months','readiness_score','readiness_tier']]
print(f'\nActive residents by readiness score ({len(active)} total):')
print(active.head(10).to_string(index=False))

**Business Interpretation:**

- **Risk improvement** is the strongest signal — residents whose risk classification has decreased over time are more likely to successfully reintegrate. This validates the case management process: consistent monitoring *does* work.
- **Plan completion rate** is highly actionable — residents who complete structured intervention goals succeed more often. Social workers should track open plans actively.
- **Session count and progress ratio** — more counseling with noted progress correlates with success. Disengaged residents (few sessions, few progress notes) should be flagged early.
- **Incident count** is negatively predictive — high-incident residents need additional stabilization before reintegration is appropriate.

**Error cost:**
- False positive (flagging not-ready as ready): **High cost** — could expose a vulnerable girl to harm. This score is a *decision support tool only*. Social worker judgment overrides the model.
- False negative (missing a ready resident): Medium cost — resident stays longer than necessary.

## 5. Causal and Relationship Analysis

**Explanatory findings from OLS logistic (causal frame):**

| Feature | Odds Ratio Direction | Causal Defensibility |
|---|---|---|
| Risk improvement | OR > 1 (positive) | Directional; partially confounded by social worker judgment |
| Plan completion rate | OR > 1 (positive) | **Most defensible**: completing goals is a structured, actionable intervention |
| Incident count | OR < 1 (negative) | Strong signal; likely bidirectional (instability causes incidents, incidents cause more instability) |
| Session count | OR > 1 (positive) | Confounded by selection — engaged residents may differ from disengaged ones |
| Avg health score | OR > 1 (positive) | Plausible; physical health supports psychological readiness |

**What we cannot claim causally:**
- Whether *more* counseling sessions *cause* better outcomes, vs. better-progressing residents simply having more sessions recorded.
- Whether length of stay causally affects outcomes, vs. longer-staying residents being systematically more complex cases.

**Dataset size warning:** With ~60 residents, no individual coefficient should be interpreted with high confidence. These are directional signals to validate as the organization grows. The prediction/explanation distinction still matters at this scale — we should favor interpretable coefficients over black-box accuracy here because the stakes of false positives are very high.

**Defensible recommendation:** Prioritize plan completion tracking. Staff should actively help residents close open intervention plans — this is the most controllable lever and has the clearest mechanistic pathway to readiness.

## 6. Deployment Notes

**Web app integration:**
- `GET /api/ml/reintegration-readiness` — returns scores for all active residents
- `GET /api/ml/reintegration-readiness/{residentId}` — score for one resident
- **Caseload Inventory page:** Readiness badge (Not Ready / Approaching / Ready) on each resident row
- **Resident detail page:** Score displayed prominently with disclaimer: *"Readiness Indicator — Social Worker Assessment Required"*

**Important UX note:** The score should never be presented as a final decision. UI copy must make clear this is one input among many.

**Model artifact:** `ml-pipelines/reintegration_readiness_model.sav`

In [ ]:
import joblib

best_gb.fit(feat_df.drop(columns=['reintegration_success']),
            feat_df['reintegration_success'])
joblib.dump(best_gb, 'reintegration_readiness_model.sav')

model_df[['resident_id','case_status','readiness_score','readiness_tier']]\
  .to_csv('reintegration_readiness_scores.csv', index=False)

print('Model saved: reintegration_readiness_model.sav')
print('Scores saved: reintegration_readiness_scores.csv')